# 05. Unsupervised Machine Learning Customer Clustering (K-Means)

## Objective
Apply unsupervised **K-Means Clustering** on Recency, Frequency, and Monetary features as an ML complement to rule-based RFM scoring.

### Steps:
1. Inspect skewness of raw RFM features.
2. Apply `log1p` transformation to reduce extreme right skew.
3. Standardize features using `StandardScaler`.
4. Determine optimal number of clusters $K$ using Elbow Method & Silhouette Score.
5. Train K-Means model with $K=4$.
6. Profile ML clusters and compare against rule-based RFM segments.

### Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print("ML libraries loaded.")

### Step 2: Load RFM Customer Dataset

In [ ]:
rfm = pd.read_csv('../data/processed/rfm_customers.csv')
rfm.head()

### Step 3: Check Skewness of Raw RFM Metrics
Highly right-skewed distributions adversely affect Euclidean distance metrics in K-Means.

In [ ]:
skewness = rfm[['Recency', 'Frequency', 'Monetary']].skew()
print("Feature Skewness:")
print(skewness)

### Step 4: Apply Log Transformation (`log1p`)
Normalize distributions using `np.log1p`.

In [ ]:
rfm_log = pd.DataFrame()
rfm_log['Recency'] = np.log1p(rfm['Recency'])
rfm_log['Frequency'] = np.log1p(rfm['Frequency'])
rfm_log['Monetary'] = np.log1p(rfm['Monetary'].clip(lower=0))

print("Skewness after Log Transformation:")
print(rfm_log.skew())

### Step 5: Feature Standardisation (`StandardScaler`)
Standardize log-transformed features to zero mean and unit variance.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_log)

scaled_df = pd.DataFrame(X_scaled, columns=['Recency_Scaled', 'Frequency_Scaled', 'Monetary_Scaled'])
scaled_df.head()

### Step 6: Evaluate Optimal Cluster Count K (Elbow & Silhouette)
Test $K \in [2, 8]$ and compute Inertia and Silhouette Scores.

In [ ]:
k_results = []
for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertia = kmeans.inertia_
    sil = silhouette_score(X_scaled, labels)
    k_results.append({'K': k, 'Inertia': round(inertia, 2), 'Silhouette_Score': round(sil, 4)})

eval_df = pd.DataFrame(k_results)
eval_df

### Step 7: Visualization — Elbow Method & Silhouette Curve

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))

color = 'tab:blue'
ax1.set_xlabel('Number of Clusters (K)', fontsize=11)
ax1.set_ylabel('Inertia (Elbow)', color=color, fontsize=11)
ax1.plot(eval_df['K'], eval_df['Inertia'], marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Silhouette Score', color=color, fontsize=11)
ax2.plot(eval_df['K'], eval_df['Silhouette_Score'], marker='s', color=color, linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('K-Means Cluster Evaluation (Elbow & Silhouette)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### Step 8: Train K-Means Model with K=4
Fit final KMeans model and assign cluster labels.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(X_scaled)
rfm['Cluster_Label'] = rfm['Cluster'].apply(lambda c: f"Cluster {c}")
rfm.head()

### Step 9: Profile Discovered ML Clusters
Compute average Recency, Frequency, Monetary, customer count, and total revenue per cluster.

In [ ]:
cluster_profiles = rfm.groupby('Cluster').agg(
    Customer_Count=('CustomerID', 'count'),
    Total_Revenue=('Monetary', 'sum'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).reset_index()

cluster_profiles['Revenue_%'] = (cluster_profiles['Total_Revenue'] / rfm['Monetary'].sum() * 100).round(2)
cluster_profiles

### Step 10: Cross-Tabulation — RFM Segments vs K-Means Clusters
Compare rule-based RFM segments against unsupervised ML clusters.

In [ ]:
cross_tab = pd.crosstab(rfm['Segment'], rfm['Cluster_Label'], margins=True)
cross_tab

### Step 11: Save Final Segmented & Clustered Dataset
Export final dataset to `../data/processed/customer_segments.csv`.

In [ ]:
rfm.to_csv('../data/processed/customer_segments.csv', index=False)
print("Final segmented and clustered dataset saved to ../data/processed/customer_segments.csv")